# Product and content

Cluster feedback, check copy against the voice guide, and look at translations and alt text. Jev still does not write the copy. One cell optionally asks OpenAI for a sentence, then checks it.


In [ ]:
import sys
from datetime import date
from pathlib import Path
import json
import re
import statistics

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from langchain_typesafe import Choice, Noul, NoulCriteria, Score
from jev_examples.settings import ask, ask_many, draft, jev_model, openai_ready, show, typesafe_ready
from jev_examples.sample_data import (
    corpus_docs,
    customers,
    emails,
    load_json,
    lookup_order,
    open_incidents,
    order,
    products,
    read_text,
    ticket,
    tickets,
)

print("Jev model:", jev_model())
print("Jev key set:", typesafe_ready())
print("OpenAI key set:", openai_ready())


## 54. Cluster feedback into a fixed list of themes

`other` is a real bucket. A growing pile of `other` means you have a theme you have not named yet.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    themes = {
        "pack_quality": "Straps, rain covers, zippers, durability",
        "checkout": "Paying, logging in, or the website",
        "shipping": "Delivery time or tracking",
        "praise": "A happy note with no problem",
        "other": "Does not fit the themes above",
    }
    questions = {
        "theme": Choice(instructions="Which theme does `feedback` belong to?", criteria=themes),
        "sentiment": Score(
            instructions="How does the writer feel?",
            criteria=["Negative", "Neutral", "Positive"],
        ),
    }
    items = load_json("feedback.json")
    requests = [{"state": {"feedback": text}, "questions": questions} for text in items]
    emerging = []
    for text, response in zip(items, ask_many(requests)):
        show(response)
        answer = response.choices["theme"]
        if answer.choice == "other" or answer.confidence < 0.5:
            emerging.append(text)
        print(answer.choice, round(response.scores["sentiment"].score, 2), text)
    print("emerging:", emerging)


**What you should see.** The rain-cover note should be pack quality and negative. The early tent should be praise or shipping, and positive. 'Save my helmet size' may be other.


## 55. Brand voice and promises

The voice guide is `brand-voice.md`. Superlatives, competitor names, and promised dates go to legal. A calm sentence does not.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    voice = read_text("brand-voice.md")
    questions = {
        "on_brand": Noul(instructions="Does `copy` follow the tone in `voice_guide`?"),
        "makes_promise": Noul(instructions="Does `copy` promise a date, a guarantee, or a ranking?"),
        "mentions_competitor": Noul(instructions="Does `copy` name a competitor?"),
    }
    for text in load_json("content.json")["copy"]:
        response = ask({"copy": text, "voice_guide": voice}, questions)
        show(response)
        needs_legal = response.nouls["makes_promise"].noul > 0.5 or response.nouls["mentions_competitor"].noul > 0.5
        print("on_brand", response.nouls["on_brand"].noul > 0.6, "legal", needs_legal)


**What you should see.** The rain-cover sentence should be on brand. The 'fastest and best' sentence that names RidgeLine should need legal.


## needs OpenAI — write one sentence, then check it

Optional. If the OpenAI key is missing, this cell skips the draft and does not invent a second copy check.


In [ ]:
if not openai_ready() or not typesafe_ready():
    print("skipped: this cell needs both keys")
else:
    voice = read_text("brand-voice.md")
    text = draft("Write one calm sentence about the 40L pack rain cover. Do not say best, fastest, or only. Do not name another brand.")
    print(text)
    response = ask(
        {"copy": text, "voice_guide": voice},
        {"on_brand": Noul(instructions="Does `copy` follow the tone in `voice_guide`?")},
    )
    show(response)


**What you should see.** With both keys, you get one sentence and a single on-brand probability. Without them, the cell prints skipped.


## 56. Localization check

Meaning, tone, and placeholders. A missing `{order_id}` fails even if the Spanish is otherwise fine.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "meaning_preserved": Noul(instructions="Does `translation` preserve the meaning of `source`?"),
        "placeholders_intact": Noul(instructions="Are all placeholders such as {order_id} from `source` present in `translation`?"),
    }
    pairs = load_json("content.json")["translations"]
    requests = [{"state": pair, "questions": questions} for pair in pairs]
    for pair, response in zip(pairs, ask_many(requests)):
        show(response)
        worst = min(answer.noul for answer in response.nouls.values())
        print("review" if worst < 0.7 else "ok", pair["translation"])


**What you should see.** The translation that keeps `{order_id}` should pass. The one that drops the placeholder should be reviewed.


## 57. Is the alt text enough?

Jev is text only, so the image is a written description. 'image' is not a description.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "describes_content": Noul(instructions="Does `alt_text` convey what matters in `image_description`?"),
        "too_generic": Noul(instructions="Is `alt_text` a generic word such as image, photo, or graphic?"),
    }
    for item in load_json("content.json")["alt_text"]:
        response = ask(item, questions)
        show(response)
        ok = response.nouls["describes_content"].noul > 0.65 and response.nouls["too_generic"].noul < 0.35
        print("ok" if ok else "fix", item["alt_text"])


**What you should see.** The green backpack sentence should be ok. The word `image` should be fixed.


## 58. How is a feature rollout landing?

Only tickets that are actually about the new helmet-size feature count. Python averages those sentiment scores.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    feature = load_json("content.json")["feature_description"]
    questions = {
        "mentions_feature": Noul(instructions="Does `ticket` refer to the behavior in `feature_description`?"),
        "sentiment": Score(
            instructions="How does the shopper feel about that feature?",
            criteria=["Angry", "Confused", "Neutral", "Happy"],
        ),
    }
    tickets = load_json("content.json")["feature_tickets"]
    requests = [
        {"state": {"ticket": text, "feature_description": feature}, "questions": questions}
        for text in tickets
    ]
    scores = []
    for text, response in zip(tickets, ask_many(requests)):
        show(response)
        if response.nouls["mentions_feature"].noul > 0.55:
            scores.append(response.scores["sentiment"].score)
            print("about the feature:", text)
    print("mean sentiment:", round(statistics.mean(scores), 2) if scores else "no mentions")
    print("route:", "hold" if scores and statistics.mean(scores) < 1.5 else "continue")


**What you should see.** The helmet-size notes should count. The tent-tracking note should not. The mix of confused and happy may land near the hold line.
